# Per-jurisdiction Clustering & Deduplication

**Scope:** Stage 5A only. Within each jurisdiction, identify and merge near-duplicate domain classes generated independently across chunks.






In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'sentence-transformers', 'rdflib', 'scikit-learn'], check=False)
print('Dependencies ready.')

In [ ]:
import json, re, csv
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from rdflib import Graph, URIRef, Literal, Namespace, RDF, RDFS, OWL
from rdflib.namespace import SKOS
print('Imports loaded.')

In [ ]:
# -- Stage 5A FULL RUN configuration ----------------------------------------
BASE_DIR = Path('/Users/umair/Synthesising Regulatory Ontologies')

# INPUT paths (UPDATED for full run)
STAGE2_MANIFEST = BASE_DIR / '3 - Extraction and  Validation Layer' / 'output' / 'stage2_extraction' / 'stage2_manifest.json'
STAGE3_5_REVALIDATION = BASE_DIR / '3 - Extraction and  Validation Layer' / 'output' / 'stage3_5_corrected' / 'stage3_5_revalidation_manifest.json'
STAGE3_5_TTL_DIR = BASE_DIR / '3 - Extraction and  Validation Layer' / 'output' / 'stage3_5_corrected'

# OUTPUT paths
OUTPUT_DIR = BASE_DIR / '4 - Consolidation and Evaluation Layer' / 'output' / 'stage5a_dedup'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEDUP_MANIFEST = OUTPUT_DIR / 'stage5a_dedup_manifest.json'
CANONICAL_CLASSES = OUTPUT_DIR / 'stage5a_canonical_classes.json'
MERGE_REVIEW_CSV = OUTPUT_DIR / 'merge_decisions_for_review.csv'
TTL_DIR = OUTPUT_DIR / 'consolidated_ttl'
TTL_DIR.mkdir(parents=True, exist_ok=True)

# Embedding model
EMBED_MODEL = 'BAAI/bge-m3'

# Similarity threshold (UPDATED based on pilot results)
SIM_THRESHOLD = 0.85   # ← Changed from 0.90 (pilot showed 0.90 too strict)

# Constraint: same CCO parent required for merge
SAME_PARENT_REQUIRED = True

# Verify all input paths exist
assert STAGE2_MANIFEST.exists(), f'Stage 2 manifest not found: {STAGE2_MANIFEST}'
assert STAGE3_5_REVALIDATION.exists(), f'Stage 3.5 manifest not found: {STAGE3_5_REVALIDATION}'
assert STAGE3_5_TTL_DIR.exists(), f'Stage 3.5 TTL directory not found: {STAGE3_5_TTL_DIR}'

print(f' All input paths verified')
print(f'Stage 2 manifest:    {STAGE2_MANIFEST.name}')
print(f'Stage 3.5 manifest:  {STAGE3_5_REVALIDATION.name}')
print(f'Stage 3.5 TTL dir:   {STAGE3_5_TTL_DIR.name}')
print(f'Output dir:          {OUTPUT_DIR}')
print(f'Embedding model:     {EMBED_MODEL}')
print(f'Similarity threshold: {SIM_THRESHOLD}')
print(f'Same parent required: {SAME_PARENT_REQUIRED}')

In [ ]:
# -- Load Stage 2 manifest + Stage 3.5 passing list -------------------------
print('Loading Stage 2 manifest...')
with open(STAGE2_MANIFEST, encoding='utf-8') as f:
    stage2 = json.load(f)
print(f'  Stage 2 chunks: {len(stage2["results"])}')

print('Loading Stage 3.5 revalidation manifest...')
with open(STAGE3_5_REVALIDATION, encoding='utf-8') as f:
    stage3_5 = json.load(f)

# Get passing chunks from Stage 3.5 (95.8% pass rate = ~1128 chunks)
passed_uids = set(r['unit_id'] for r in stage3_5['results'] if r['pass'])
print(f'  Stage 3.5 passing chunks: {len(passed_uids)}')

# Filter Stage 2 results to passing chunks
passing_chunks = [r for r in stage2['results'] if r['unit_id'] in passed_uids]
print(f'  Final passing chunks for consolidation: {len(passing_chunks)}')

# Per-jurisdiction breakdown
jur_counts = Counter(r['jurisdiction_prefix'] for r in passing_chunks)
print('\nPer-jurisdiction passing chunks:')
for jur, count in sorted(jur_counts.items()):
    print(f'  {jur}: {count}')

In [ ]:
# -- Read corrected TTLs from Stage 3.5 (NOT Stage 2 raw output!) ----------
def load_corrected_ttl(unit_id):
    """Load corrected TTL from Stage 3.5 output."""
    ttl_path = STAGE3_5_TTL_DIR / f'{unit_id}.ttl'
    if not ttl_path.exists():
        return None
    return ttl_path.read_text(encoding='utf-8')

# Verify a sample TTL loads correctly
sample_uid = passing_chunks[0]['unit_id']
sample_ttl = load_corrected_ttl(sample_uid)
if sample_ttl:
    print(f' Sample corrected TTL loaded: {sample_uid}')
    print(f'  Length: {len(sample_ttl)} chars')
else:
    print(f' Failed to load TTL for {sample_uid}')
    raise FileNotFoundError(f'Stage 3.5 TTL missing for {sample_uid}')

In [ ]:
# -- Extract domain class declarations per jurisdiction --------------------
def extract_classes(ttl: str, jur_prefix: str):
    """Extract all gro-XX:ClassName declarations with parent, label, definition."""
    pattern = re.compile(
        rf'({jur_prefix}:[A-Z]\w+)\s+a\s+owl:Class\s*;\s*rdfs:subClassOf\s+(\S+)\s*;\s*rdfs:label\s+"([^"]*)"\s*(?:;\s*skos:definition\s+"([^"]*)")?',
        re.DOTALL
    )
    out = []
    for m in pattern.finditer(ttl):
        out.append({
            'class':      m.group(1),
            'parent':     m.group(2).rstrip(';.,'),
            'label':      m.group(3),
            'definition': (m.group(4) or '').strip(),
        })
    return out

print('Extracting class declarations from corrected TTLs...')
classes_by_jur = defaultdict(list)
ttl_load_errors = []

for r in passing_chunks:
    jur_prefix = r['jurisdiction_prefix']
    corrected_ttl = load_corrected_ttl(r['unit_id'])
    
    if corrected_ttl is None:
        ttl_load_errors.append(r['unit_id'])
        continue
    
    for cls in extract_classes(corrected_ttl, jur_prefix):
        cls['source_chunk'] = r['unit_id']
        classes_by_jur[jur_prefix].append(cls)

print(f'\n Processed {len(passing_chunks) - len(ttl_load_errors)} chunks')
if ttl_load_errors:
    print(f'⚠ TTL load errors: {len(ttl_load_errors)}')

print('\nClass population per jurisdiction (with duplicates):')
total_raw = 0
for j in sorted(classes_by_jur):
    n = len(classes_by_jur[j])
    total_raw += n
    print(f'  {j}: {n} class declarations')
print(f'  TOTAL: {total_raw}')

In [ ]:
# -- Step 1: Trivial dedup — exact URI matches collapse first --------------
print('STEP 1: Trivial deduplication (exact URI matches)')
print('=' * 60)

dedup_trivial = {}
for jur, classes in classes_by_jur.items():
    seen = {}
    for c in classes:
        uri = c['class']
        if uri not in seen:
            seen[uri] = dict(c)
            seen[uri]['source_chunks'] = [c['source_chunk']]
            del seen[uri]['source_chunk']
        else:
            seen[uri]['source_chunks'].append(c['source_chunk'])
            # Prefer the longest definition
            if len(c['definition']) > len(seen[uri]['definition']):
                seen[uri]['definition'] = c['definition']
    dedup_trivial[jur] = list(seen.values())

print('\nAfter trivial dedup:')
total_trivial = 0
for j in sorted(dedup_trivial):
    n_in = len(classes_by_jur[j])
    n_out = len(dedup_trivial[j])
    total_trivial += n_out
    print(f'  {j}: {n_in} -> {n_out} (collapsed {n_in - n_out} exact duplicates)')
print(f'  TOTAL: {total_raw} -> {total_trivial} (-{total_raw - total_trivial})')

In [ ]:
# -- Step 2: Embed surviving classes (label + definition) ------------------
print('STEP 2: Embedding classes for semantic similarity')
print('=' * 60)
print(f'Loading {EMBED_MODEL}...')
model = SentenceTransformer(EMBED_MODEL)
print(f'  Embedding dim: {model.get_sentence_embedding_dimension()}')

embeddings_by_jur = {}
for jur, classes in dedup_trivial.items():
    texts = [f"{c['label']}. {c['definition']}" for c in classes]
    print(f'  Embedding {jur}: {len(texts)} classes...')
    emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=False, batch_size=64)
    embeddings_by_jur[jur] = emb

print(' Embeddings computed for all jurisdictions')

In [ ]:
# -- Step 3: Compute pairwise similarity within each jurisdiction ----------
print(f'STEP 3: Finding merge candidates (threshold={SIM_THRESHOLD})')
print('=' * 60)

merge_candidates_by_jur = {}

for jur, classes in dedup_trivial.items():
    if len(classes) < 2:
        merge_candidates_by_jur[jur] = []
        continue
    
    sim = cosine_similarity(embeddings_by_jur[jur])
    
    candidates = []
    for i in range(len(classes)):
        for j in range(i+1, len(classes)):
            if SAME_PARENT_REQUIRED and classes[i]['parent'] != classes[j]['parent']:
                continue
            s = float(sim[i, j])
            if s >= SIM_THRESHOLD:
                candidates.append({
                    'similarity': round(s, 4),
                    'class_a':    classes[i]['class'],
                    'class_b':    classes[j]['class'],
                    'parent':     classes[i]['parent'],
                    'label_a':    classes[i]['label'],
                    'label_b':    classes[j]['label'],
                    'definition_a': classes[i]['definition'],
                    'definition_b': classes[j]['definition'],
                    'i': i, 'j': j,
                })
    candidates.sort(key=lambda x: x['similarity'], reverse=True)
    merge_candidates_by_jur[jur] = candidates

print('\nMerge candidates per jurisdiction:')
total_candidates = 0
for jur in sorted(merge_candidates_by_jur):
    cands = merge_candidates_by_jur[jur]
    total_candidates += len(cands)
    print(f'  {jur}: {len(cands)} candidate pair(s)')
    for c in cands[:5]:  # Show top 5 per jurisdiction
        print(f'    [{c["similarity"]:.3f}] {c["class_a"]} <-> {c["class_b"]}')
        print(f'             A: "{c["label_a"]}"')
        print(f'             B: "{c["label_b"]}"')
        print(f'             parent: {c["parent"]}')

print(f'\nTOTAL merge candidates across all jurisdictions: {total_candidates}')

In [ ]:
# -- Step 4 (NEW): Export merge decisions for manual review -----------------
print('STEP 4: Exporting merge candidates to CSV for manual review')
print('=' * 60)

with open(MERGE_REVIEW_CSV, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'jurisdiction', 'similarity', 'class_a', 'label_a', 'definition_a',
        'class_b', 'label_b', 'definition_b', 'parent',
        'recommendation', 'approve_merge'
    ])
    writer.writeheader()
    
    for jur, cands in sorted(merge_candidates_by_jur.items()):
        for c in cands:
            # Auto-recommendation based on similarity
            if c['similarity'] >= 0.92:
                rec = 'STRONG MERGE (high confidence)'
            elif c['similarity'] >= 0.88:
                rec = 'LIKELY MERGE (review labels/definitions)'
            else:
                rec = 'REVIEW CAREFULLY (low confidence)'
            
            writer.writerow({
                'jurisdiction': jur,
                'similarity':   c['similarity'],
                'class_a':      c['class_a'],
                'label_a':      c['label_a'],
                'definition_a': c['definition_a'],
                'class_b':      c['class_b'],
                'label_b':      c['label_b'],
                'definition_b': c['definition_b'],
                'parent':       c['parent'],
                'recommendation': rec,
                'approve_merge': 'YES',  # Default — change to NO to reject
            })

print(f' Merge decisions exported: {MERGE_REVIEW_CSV}')
print(f'\n MANUAL REVIEW INSTRUCTION:')
print(f'  1. Open: {MERGE_REVIEW_CSV}')
print(f'  2. Review each row\'s "approve_merge" column')
print(f'  3. Change "YES" to "NO" for rows you want to reject')
print(f'  4. Save the file')
print(f'  5. Run Cell 11 (next cell) to apply approved merges only')
print(f'\nOR proceed with all merges (skip review) by running Cell 11 directly.')

In [ ]:
# -- Step 5: Apply approved merges from CSV (read approval column) ----------
print('STEP 5: Applying approved merges')
print('=' * 60)

# Re-read CSV to get approval decisions
approved_pairs = defaultdict(set)
total_approved = 0
total_rejected = 0

if MERGE_REVIEW_CSV.exists():
    with open(MERGE_REVIEW_CSV, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            approve = row['approve_merge'].strip().upper()
            if approve == 'YES':
                approved_pairs[row['jurisdiction']].add((row['class_a'], row['class_b']))
                total_approved += 1
            else:
                total_rejected += 1
    print(f' Approved merges: {total_approved}')
    print(f' Rejected merges: {total_rejected}')
else:
    # Fallback: approve all
    for jur, cands in merge_candidates_by_jur.items():
        approved_pairs[jur] = {(c['class_a'], c['class_b']) for c in cands}
        total_approved += len(cands)
    print(f' No review CSV found, auto-approving all {total_approved} merges')

In [ ]:
# -- Step 6: Cluster approved merges into equivalence classes ---------------
def union_find_merge(n, pairs):
    """Return parent[] where same parent = same cluster."""
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    for i, j in pairs:
        union(i, j)
    return [find(x) for x in range(n)]

canonical_by_jur = {}
merge_map_by_jur = {}

for jur, classes in dedup_trivial.items():
    # Build pairs from APPROVED merges only
    approved = approved_pairs.get(jur, set())
    
    # Get indices for approved class pairs
    pairs = []
    for cand in merge_candidates_by_jur[jur]:
        if (cand['class_a'], cand['class_b']) in approved:
            pairs.append((cand['i'], cand['j']))
    
    if not pairs:
        # No approved merges, keep all as canonical
        canonical_by_jur[jur] = classes
        merge_map_by_jur[jur] = {c['class']: c['class'] for c in classes}
        continue
    
    cluster_ids = union_find_merge(len(classes), pairs)
    clusters = defaultdict(list)
    for idx, cid in enumerate(cluster_ids):
        clusters[cid].append(idx)
    
    canonical = []
    mmap = {}
    for cid, indices in clusters.items():
        cluster_classes = [classes[i] for i in indices]
        cluster_classes.sort(key=lambda c: (-len(c['definition']), c['class']))
        canon = dict(cluster_classes[0])
        canon['merged_from'] = [c['class'] for c in cluster_classes]
        canon['merged_source_chunks'] = sorted(set(
            ch for c in cluster_classes for ch in c.get('source_chunks', [])
        ))
        canonical.append(canon)
        for c in cluster_classes:
            mmap[c['class']] = canon['class']
    
    canonical_by_jur[jur] = canonical
    merge_map_by_jur[jur] = mmap

print('\nAfter clustering:')
total_canon = 0
for jur in sorted(canonical_by_jur):
    n_in = len(dedup_trivial[jur])
    n_out = len(canonical_by_jur[jur])
    total_canon += n_out
    print(f'  {jur}: {n_in} -> {n_out} canonical classes (collapsed {n_in-n_out})')
print(f'  TOTAL: {total_trivial} -> {total_canon}')

# Show clusters with >1 member
print('\nMulti-member clusters (semantic merges):')
n_multi = 0
for jur, canons in sorted(canonical_by_jur.items()):
    for c in canons:
        if len(c.get('merged_from', [c['class']])) > 1:
            n_multi += 1
            print(f'  {jur}: canonical {c["class"]}')
            for src in c['merged_from']:
                if src != c['class']:
                    print(f'         absorbed: {src}')
if n_multi == 0:
    print('  (none)')

In [ ]:
# -- Step 7: Build consolidated per-jurisdiction TTL files ------------------
print('STEP 7: Building consolidated TTL files')
print('=' * 60)

PREFIX_BASE = {
    'gro-uk': 'https://w3id.org/cco-gro/onto/uk#',
    'gro-us': 'https://w3id.org/cco-gro/onto/us#',
    'gro-ca': 'https://w3id.org/cco-gro/onto/ca#',
    'gro-au': 'https://w3id.org/cco-gro/onto/au#',
    'gro':    'https://w3id.org/cco-gro/onto#',
    'cco':    'https://www.w3id.org/cco/cco#',
    'data':   'https://w3id.org/cco-gro/data#',
}

def qname_to_uri(qname: str) -> URIRef:
    if ':' in qname:
        pfx, local = qname.split(':', 1)
        if pfx in PREFIX_BASE:
            return URIRef(PREFIX_BASE[pfx] + local)
    return URIRef(qname)

def render_canonical_classes(jur, canonical_list):
    g = Graph()
    for pfx, uri in PREFIX_BASE.items():
        g.bind(pfx, uri)
    g.bind('skos', SKOS)
    g.bind('owl', OWL)
    g.bind('rdfs', RDFS)
    for c in canonical_list:
        cls_uri = qname_to_uri(c['class'])
        parent_uri = qname_to_uri(c['parent'])
        g.add((cls_uri, RDF.type, OWL.Class))
        g.add((cls_uri, RDFS.subClassOf, parent_uri))
        g.add((cls_uri, RDFS.label, Literal(c['label'])))
        if c['definition']:
            g.add((cls_uri, SKOS.definition, Literal(c['definition'])))
    return g

def build_uri_merge_map(qname_map):
    uri_map = {}
    for old_q, new_q in qname_map.items():
        if old_q == new_q: continue
        uri_map[qname_to_uri(old_q)] = qname_to_uri(new_q)
    return uri_map

def rewrite_term(term, uri_map):
    if isinstance(term, URIRef) and term in uri_map:
        return uri_map[term]
    return term

# Build consolidated graph per jurisdiction
for jur, canonical_list in canonical_by_jur.items():
    chunks_in_jur = [r for r in passing_chunks if r['jurisdiction_prefix'] == jur]
    uri_map = build_uri_merge_map(merge_map_by_jur[jur])
    
    consolidated_g = render_canonical_classes(jur, canonical_list)
    
    parse_errors = 0
    for r in chunks_in_jur:
        #  CRITICAL CHANGE: Use Stage 3.5 corrected TTLs, NOT Stage 2 raw
        corrected_ttl = load_corrected_ttl(r['unit_id'])
        if corrected_ttl is None:
            continue
        
        chunk_g = Graph()
        try:
            chunk_g.parse(data=corrected_ttl, format='turtle')
        except Exception as e:
            print(f'  WARN: failed to parse {r["unit_id"]}: {str(e)[:80]}')
            parse_errors += 1
            continue
        
        class_uris = set(chunk_g.subjects(RDF.type, OWL.Class))
        
        for s, p, o in chunk_g:
            if s in class_uris:
                continue
            s2 = rewrite_term(s, uri_map)
            p2 = rewrite_term(p, uri_map)
            o2 = rewrite_term(o, uri_map)
            consolidated_g.add((s2, p2, o2))
    
    # Bind prefixes
    for pfx, uri in PREFIX_BASE.items():
        consolidated_g.bind(pfx, uri)
    consolidated_g.bind('skos', SKOS)
    consolidated_g.bind('prov', Namespace('http://www.w3.org/ns/prov#'))
    consolidated_g.bind('xsd',  Namespace('http://www.w3.org/2001/XMLSchema#'))
    
    out_path = TTL_DIR / f'consolidated_{jur}.ttl'
    consolidated_g.serialize(destination=str(out_path), format='turtle')
    
    # Verify it parses back
    try:
        verify = Graph()
        verify.parse(str(out_path), format='turtle')
        status = f'OK ({len(verify)} triples)'
    except Exception as e:
        status = f'PARSE ERROR: {str(e)[:80]}'
    
    error_note = f' [{parse_errors} parse errors]' if parse_errors > 0 else ''
    print(f'  {out_path.name}: {status}{error_note}')

In [ ]:
# -- Save manifests ----------------------------------------------------------
manifest = {
    'metadata': {
        'run_type':              'stage5a_full_run',
        'stage':                 'stage5a',
        'embed_model':           EMBED_MODEL,
        'sim_threshold':         SIM_THRESHOLD,
        'same_parent_required':  SAME_PARENT_REQUIRED,
        'created_at':            datetime.now().isoformat(),
        'stage2_source':         str(STAGE2_MANIFEST),
        'stage3_5_source':       str(STAGE3_5_REVALIDATION),
        'stage3_5_ttl_dir':      str(STAGE3_5_TTL_DIR),
        'total_input_chunks':    len(passing_chunks),
        'total_approved_merges': total_approved,
        'total_rejected_merges': total_rejected,
    },
    'per_jurisdiction': {
        jur: {
            'n_raw_declarations':    len(classes_by_jur[jur]),
            'n_after_trivial_dedup': len(dedup_trivial[jur]),
            'n_canonical':           len(canonical_by_jur[jur]),
            'merge_candidates':      merge_candidates_by_jur[jur],
            'merge_map':             merge_map_by_jur[jur],
        }
        for jur in sorted(classes_by_jur)
    },
}

with open(DEDUP_MANIFEST, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(f' Dedup manifest saved: {DEDUP_MANIFEST}')

canonical_payload = {
    'metadata': {
        'created_at': datetime.now().isoformat(),
        'source':     str(DEDUP_MANIFEST),
    },
    'per_jurisdiction': {
        jur: canonical_by_jur[jur]
        for jur in sorted(canonical_by_jur)
    },
}

with open(CANONICAL_CLASSES, 'w', encoding='utf-8') as f:
    json.dump(canonical_payload, f, indent=2, ensure_ascii=False)
print(f' Canonical classes saved: {CANONICAL_CLASSES}')

In [ ]:
# -- Final summary -----------------------------------------------------------
print('=' * 70)
print('STAGE 5A FULL RUN SUMMARY')
print('=' * 70)
print(f'{"Jurisdiction":12s} {"Raw":>5s} {"Trivial":>8s} {"Canonical":>10s} {"Reduction":>10s}')
print('-' * 50)

total_raw_all = 0
total_canon_all = 0

for jur in sorted(classes_by_jur):
    n_raw   = len(classes_by_jur[jur])
    n_triv  = len(dedup_trivial[jur])
    n_canon = len(canonical_by_jur[jur])
    reduction = n_raw - n_canon
    total_raw_all += n_raw
    total_canon_all += n_canon
    print(f'{jur:12s} {n_raw:>5d} {n_triv:>8d} {n_canon:>10d} {reduction:>10d}')

print('-' * 50)
print(f'{"TOTAL":12s} {total_raw_all:>5d} {"":>8s} {total_canon_all:>10d} {total_raw_all-total_canon_all:>10d}')
print(f'\nOverall reduction: {(1 - total_canon_all/total_raw_all)*100:.1f}%')

print(f'\nMerge decisions:')
print(f'  Approved: {total_approved}')
print(f'  Rejected: {total_rejected}')

print(f'\nOutput files:')
print(f'  TTL files:        {TTL_DIR}/')
print(f'  Dedup manifest:   {DEDUP_MANIFEST.name}')
print(f'  Canonical:        {CANONICAL_CLASSES.name}')
print(f'  Review CSV:       {MERGE_REVIEW_CSV.name}')

print(f'\nNext step: Stage 5B (cross-jurisdiction alignment)')
print(f'OR Stage 5E.1 (Quality Metrics computation)')